# Canonical validation protocol

This notebook inspects the immutable artifact produced by `scripts/prepare_canonical_data.py`. The split, aggregation, ground-truth construction, and validation logic live in importable Python modules; they are not duplicated here.

In [ ]:
import json
from pathlib import Path

import polars as pl

from data_utils import (
    DAILY_INTERACTION_SCHEMA,
    GROUND_TRUTH_SCHEMA,
    TARGET_USER_SCHEMA,
)

pl.Config.set_tbl_rows(30)

In [ ]:
artifact_dir = Path("artifacts/task01_canonical_data_v1")
config = json.loads((artifact_dir / "config.json").read_text())
metrics = json.loads((artifact_dir / "metrics.json").read_text())
diagnostics = metrics["deterministic_diagnostics"]

config["split"], diagnostics["split"]

In [ ]:
pl.DataFrame(
    [
        {"side": side, **values}
        for side, values in diagnostics["daily"].items()
    ]
)

In [ ]:
pl.DataFrame(
    {
        "stage": list(diagnostics["ground_truth_funnel"]),
        "value": list(diagnostics["ground_truth_funnel"].values()),
    },
    strict=False,
)

In [ ]:
history = pl.scan_parquet(artifact_dir / "history_daily.parquet")
validation = pl.scan_parquet(artifact_dir / "validation_daily.parquet")
ground_truth = pl.scan_parquet(artifact_dir / "ground_truth.parquet")
target_ground_truth = pl.scan_parquet(
    artifact_dir / "target_ground_truth.parquet"
)
target_users = pl.scan_parquet(artifact_dir / "target_users.parquet")

assert history.collect_schema() == DAILY_INTERACTION_SCHEMA
assert validation.collect_schema() == DAILY_INTERACTION_SCHEMA
assert ground_truth.collect_schema() == GROUND_TRUTH_SCHEMA
assert target_ground_truth.collect_schema() == GROUND_TRUTH_SCHEMA
assert target_users.collect_schema() == TARGET_USER_SCHEMA

diagnostics["output_sha256"]

## Interpretation

All canonical models fit from `history_daily.parquet`. `validation_daily.parquet` is reserved for validation and has already been converted into seen-filtered, known-item-only ground truth. Existing calendar-split snapshots under `data/` are intentionally absent from this protocol.